In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile
from rdkit import Chem
from rdkit.Chem import AllChem
import os
import nglview
import numpy as np
from openff.toolkit import ForceField, Molecule, Topology
from openff.units import unit
from openff.interchange import Interchange
from openff.interchange.components._packmol import UNIT_CUBE, RHOMBIC_DODECAHEDRON, pack_box
from openff.interchange.drivers import get_openmm_energies
from openff.interchange.drivers.all import get_summary_data

In [ ]:
ID_list = ["0091", "0272", "0482", "0506", "0567", "0736", "0782", "0811", "0822", "1654"]

#seq_type = "binder"
seq_type = "nonbinder"
#seq_type = "neg_low_pkt"
#seq_type = "neg_fail_gate"

#box_shape = "cube"
box_shape = "dodecahedron"

prefix = "pair"  # "pair", "bind", "nonb", "seq"

## Step 1: Add Hydrogens and Repair PDB (pdbfixer)

In [ ]:
for ID in ID_list:
    if seq_type == "binder":
        fixer = PDBFixer(filename=f"binders/{prefix}_{ID}_binder/protein_{prefix}{ID}.pdb")
    elif seq_type == "nonbinder":
        fixer = PDBFixer(filename=f"nonbinders/{prefix}_{ID}_nb/protein_{prefix}{ID}.pdb")
    elif seq_type == "neg_low_pkt":
        fixer = PDBFixer(filename=f"neg_low_pkt/{prefix}_{ID}_low_pkt/protein_{prefix}{ID}.pdb")
    elif seq_type == "neg_fail_gate":
        fixer = PDBFixer(filename=f"neg_fail_gate/{prefix}_{ID}_fail_gate/protein_{prefix}{ID}.pdb")

    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=7.0)

    if seq_type == "binder":
        with open(f"binders/{prefix}_{ID}_binder/protein_{prefix}{ID}_fixed_H.pdb", "w") as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)
    elif seq_type == "nonbinder":
        with open(f"nonbinders/{prefix}_{ID}_nb/protein_{prefix}{ID}_fixed_H.pdb", "w") as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)
    elif seq_type == "neg_low_pkt":
        with open(f"neg_low_pkt/{prefix}_{ID}_low_pkt/protein_{prefix}{ID}_fixed_H.pdb", "w") as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)
    elif seq_type == "neg_fail_gate":
        with open(f"neg_fail_gate/{prefix}_{ID}_fail_gate/protein_{prefix}{ID}_fixed_H.pdb", "w") as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)

    print(f"{ID}: PDB fixed and hydrogens added")

## Step 2: Convert Ligand PDB to SDF (RDKit)

In [ ]:
for ID in ID_list:
    if seq_type == "binder":
        mol = Chem.MolFromPDBFile(f"binders/{prefix}_{ID}_binder/ligand_{prefix}{ID}.pdb", removeHs=False)
        w = Chem.SDWriter(f"binders/{prefix}_{ID}_binder/ligand_{prefix}{ID}.sdf")
    elif seq_type == "nonbinder":
        mol = Chem.MolFromPDBFile(f"nonbinders/{prefix}_{ID}_nb/ligand_{prefix}{ID}.pdb", removeHs=False)
        w = Chem.SDWriter(f"nonbinders/{prefix}_{ID}_nb/ligand_{prefix}{ID}.sdf")
    elif seq_type == "neg_low_pkt":
        mol = Chem.MolFromPDBFile(f"neg_low_pkt/{prefix}_{ID}_low_pkt/ligand_{prefix}{ID}.pdb", removeHs=False)
        w = Chem.SDWriter(f"neg_low_pkt/{prefix}_{ID}_low_pkt/ligand_{prefix}{ID}.sdf")
    elif seq_type == "neg_fail_gate":
        mol = Chem.MolFromPDBFile(f"neg_fail_gate/{prefix}_{ID}_fail_gate/ligand_{prefix}{ID}.pdb", removeHs=False)
        w = Chem.SDWriter(f"neg_fail_gate/{prefix}_{ID}_fail_gate/ligand_{prefix}{ID}.sdf")

    w.write(mol)
    w.close()
    print(f"{ID}: ligand PDB converted to SDF")

## Step 3: Parameterize, Solvate, and Export to GROMACS

In [ ]:
for ID in ID_list:
    print(f"\n=== Processing {ID} ===")

    # --- Resolve paths ---
    if seq_type == "binder":
        pdb_file = os.path.abspath(f"binders/{prefix}_{ID}_binder/protein_{prefix}{ID}_fixed_H.pdb")
        sdf_file = f"binders/{prefix}_{ID}_binder/ligand_{prefix}{ID}.sdf"
        folder_name = "binders"
        extension = seq_type
    elif seq_type == "nonbinder":
        pdb_file = os.path.abspath(f"nonbinders/{prefix}_{ID}_nb/protein_{prefix}{ID}_fixed_H.pdb")
        sdf_file = f"nonbinders/{prefix}_{ID}_nb/ligand_{prefix}{ID}.sdf"
        folder_name = "nonbinders"
        extension = "nb"
    elif seq_type == "neg_low_pkt":
        pdb_file = os.path.abspath(f"neg_low_pkt/{prefix}_{ID}_low_pkt/protein_{prefix}{ID}_fixed_H.pdb")
        sdf_file = f"neg_low_pkt/{prefix}_{ID}_low_pkt/ligand_{prefix}{ID}.sdf"
        folder_name = "neg_low_pkt"
        extension = "low_pkt"
    elif seq_type == "neg_fail_gate":
        pdb_file = os.path.abspath(f"neg_fail_gate/{prefix}_{ID}_fail_gate/protein_{prefix}{ID}_fixed_H.pdb")
        sdf_file = f"neg_fail_gate/{prefix}_{ID}_fail_gate/ligand_{prefix}{ID}.sdf"
        folder_name = "neg_fail_gate"
        extension = "fail_gate"

    # --- Ligand ---
    ligand = Molecule.from_file(sdf_file)
    ligand.name = "LIG"

    ligand_intrcg = Interchange.from_smirnoff(
        force_field=ForceField("openff_unconstrained-2.0.0.offxml"),
        topology=[ligand],
    )

    # --- Protein ---
    protein_full = Topology.from_pdb(pdb_file)
    protein = protein_full.molecule(0)
    protein.name = "protein"

    ff14sb = ForceField("ff14sb_off_impropers_0.0.3.offxml")
    protein_intrcg = Interchange.from_smirnoff(
        force_field=ff14sb,
        topology=protein.to_topology(),
    )

    # --- Dock protein + ligand ---
    docked_intrcg = protein_intrcg.combine(ligand_intrcg)

    # --- Check charge and pick counterion ---
    total_charge = round(sum(docked_intrcg["Electrostatics"].charges.values()), 3)
    assert total_charge == protein.total_charge + ligand.total_charge, (
        f"Total charge mismatch: {total_charge} vs {protein.total_charge + ligand.total_charge}"
    )
    print(f"Total charge: {total_charge}")

    total_charge_e = float(total_charge.m)
    if total_charge_e < 0:
        counterion = Molecule.from_smiles("[Na+]")
        counterion.name = "NA"
        n_counterions = int(round(abs(total_charge_e)))
    elif total_charge_e > 0:
        counterion = Molecule.from_smiles("[Cl-]")
        counterion.name = "CL"
        n_counterions = int(round(abs(total_charge_e)))
    else:
        counterion = None
        n_counterions = 0

    water = Molecule.from_smiles("O")
    water.name = "SOL"
    water.generate_conformers(n_conformers=1)

    # --- Compute box and pack ---
    xyz = protein.conformers[0].to(unit.nanometer).m
    centroid = xyz.mean(axis=0)
    protein_radius_nm = np.sqrt(((xyz - centroid) ** 2).sum(axis=1).max())
    buffer_nm = 2.0
    scale_nm = 2.0 * protein_radius_nm + buffer_nm

    molecules = [water]
    number_of_copies = [None]  # filled below

    if box_shape == "cube":
        box_vectors = np.eye(3) * scale_nm * unit.nanometer
        V_box_nm3 = scale_nm ** 3
        V_solute_nm3 = (4.0 / 3.0) * np.pi * (protein_radius_nm ** 3)
        n_water = int(33.4 * max(V_box_nm3 - V_solute_nm3, 0.0))
        packmol_dir = f"/Users/ivanatang/Library/CloudStorage/OneDrive-UCB-O365/Shirts Lab/LCA_boltz_models/{folder_name}/{prefix}_{ID}_{extension}/packmol_solv_box"
        print(f"Box length: {scale_nm:.3f} nm, waters: {n_water}, counterions: {n_counterions}")

    elif box_shape == "dodecahedron":
        box_vectors = (scale_nm * RHOMBIC_DODECAHEDRON) * unit.nanometer
        box_nm = box_vectors.to(unit.nanometer).m
        V_box_nm3 = abs(np.linalg.det(box_nm))
        V_solute_nm3 = (4.0 / 3.0) * np.pi * (protein_radius_nm ** 3)
        n_water = int(33.4 * max(V_box_nm3 - V_solute_nm3, 0.0))
        packmol_dir = f"/Users/ivanatang/Library/CloudStorage/OneDrive-UCB-O365/Shirts Lab/LCA_boltz_models/{folder_name}/{prefix}_{ID}_{extension}/packmol_solv_dodecahedron"
        print(f"Protein radius: {protein_radius_nm:.3f} nm, scale: {scale_nm:.3f} nm, box vol: {V_box_nm3:.2f} nm^3")
        print(f"Waters: {n_water}, counterions: {n_counterions}")

    number_of_copies = [n_water]
    if counterion is not None and n_counterions > 0:
        molecules.append(counterion)
        number_of_copies.append(n_counterions)

    packed_topology = pack_box(
        solute=docked_intrcg.topology,
        molecules=molecules,
        number_of_copies=number_of_copies,
        box_vectors=box_vectors,
        center_solute=True,
        tolerance=2.0 * unit.angstrom,
        working_directory=packmol_dir,
        retain_working_files=True,
    )
    print(f"Total molecules packed: {packed_topology.n_molecules}")

    # --- Save packed PDB ---
    if seq_type == "binder":
        packed_topology.to_file(f"binders/{prefix}_{ID}_binder/packed_{box_shape}_{ID}.pdb")
    elif seq_type == "nonbinder":
        packed_topology.to_file(f"nonbinders/{prefix}_{ID}_nb/packed_{box_shape}_{ID}.pdb")
    elif seq_type == "neg_low_pkt":
        packed_topology.to_file(f"neg_low_pkt/{prefix}_{ID}_low_pkt/packed_{box_shape}_{prefix}{ID}.pdb")
    elif seq_type == "neg_fail_gate":
        packed_topology.to_file(f"neg_fail_gate/{prefix}_{ID}_fail_gate/packed_{box_shape}_{prefix}{ID}.pdb")

    # --- Build full solvated system ---
    topology_molecules = [water] * n_water
    if counterion is not None:
        topology_molecules += [counterion] * n_counterions

    water_intrcg = Interchange.from_smirnoff(
        force_field=ForceField("openff_unconstrained-2.0.0.offxml"),
        topology=topology_molecules,
    )

    system_intrcg = docked_intrcg.combine(water_intrcg)
    system_intrcg.positions = packed_topology.get_positions()
    system_intrcg.box = packed_topology.box_vectors

    # --- Export to GROMACS with HMR ---
    output_dir = f"/Users/ivanatang/Library/CloudStorage/OneDrive-UCB-O365/Shirts Lab/LCA_boltz_models/{folder_name}/{prefix}_{ID}_{extension}"
    os.chdir(output_dir)
    system_intrcg.to_gromacs(prefix=f"{prefix}_{ID}_{box_shape}_HMR", decimal=3, hydrogen_mass=3, monolithic=False)
    print(f"{ID}: GROMACS files written to {output_dir}")